# 🏆 Notebook 4｜迷你專案：把整章串起來（即戰力驗收）

> 對應知識地圖 **站 8**。兩個小專案，全部合成資料、全部只用前面學過的東西：
>
> - **專案 A｜紋理分類器**：3 類紋理 × 特徵向量 → kNN 分類（課本 Ch3 的機器學習最簡單入門）
> - **專案 B｜拍手聲 vs 鋼琴聲判別器**：ZCR 軌跡 + 頻譜通量 → 閾值規則
>
> 完成後，你就具備：「**給我一張圖 / 一段聲音 → 抽出關鍵特徵 → 交給分類器**」的完整基本功。

## 專案 A：紋理分類器 (kNN)

流程：
1. 合成 3 類紋理（平滑/粗糙/週期），每類 20 張（不同隨機種子 = 同一類的不同樣本）
2. 每張圖抽特徵：一階統計 5 個 + 共生矩陣 4 個（4 方向平均）+ Laws 9 個 = **18 維特徵向量**
3. 一半當訓練、一半當測試；用最簡單的 **kNN**（算距離找最近鄰居投票）分類
4. 看準確率 + 特徵散佈圖（證明特徵真的分得開類別）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
from scipy.ndimage import convolve
from skimage.feature import graycomatrix, graycoprops

# ---------- 特徵萃取（就是把 nb1 的工具全部組起來） ----------
rng = np.random.default_rng(0)

def make_texture(cls, seed):
    r = np.random.default_rng(seed)
    noise = r.standard_normal((96, 96))
    if cls == 0:                                    # 平滑
        img = ndimage.gaussian_filter(noise, sigma=7.0)
    elif cls == 1:                                  # 粗糙
        img = ndimage.gaussian_filter(noise, sigma=1.3)
    else:                                           # 週期
        yy, xx = np.indices((96, 96))
        img = np.sin(xx * 0.55) * np.sin(yy * 0.55) + 0.15 * noise
    img = (img - img.min()) / (img.max() - img.min() + 1e-12)
    return img

def features(img):
    g = np.round(img * 7).astype(int)               # 縮到 8 階（課本建議：灰階少 → 直方圖/矩陣較密）
    # 一階統計
    P, _ = np.histogram(g, bins=8, range=(0, 7), density=True)
    I = np.arange(8); m1 = (P * I).sum()
    var = (P * (I - m1) ** 2).sum(); sd = np.sqrt(var)
    skew = (P * (I - m1) ** 3).sum() / sd ** 3
    kurt = (P * (I - m1) ** 4).sum() / var ** 2 - 3
    ent = -(P[P > 0] * np.log2(P[P > 0])).sum()
    # 二階統計（4 方向平均 → 旋轉容忍）
    glcm = graycomatrix(g, distances=[1], angles=[0, 45, 90, 135],
                        levels=8, symmetric=True, normed=True)
    con = graycoprops(glcm, 'contrast')[0].mean()
    ene = graycoprops(glcm, 'energy')[0].mean()
    hom = graycoprops(glcm, 'homogeneity')[0].mean()
    cor = graycoprops(glcm, 'correlation')[0].mean()
    # Laws 9 遮罩（變異數）
    v = [np.array([1, 2, 1]) / 3, np.array([-1, 0, 1]) / 2, np.array([-1, 2, -1]) / 4]
    laws = [convolve(img, np.outer(a, b), mode='wrap').var() for a in v for b in v]
    return np.array([m1, var, skew, kurt, ent, con, ene, hom, cor] + laws)

X, y = [], []
for cls in range(3):
    for seed in range(20):
        X.append(features(make_texture(cls, seed + cls * 100)))
        y.append(cls)
X = np.array(X); y = np.array(y)
print('資料集形狀:', X.shape, '→ 60 張圖 × 18 維特徵')

In [ ]:
# ---------- kNN 分類器（手寫 8 行，無需 sklearn） ----------
def knn(Xtrain, ytrain, x, k=5):
    d = np.linalg.norm(Xtrain - x, axis=1)          # 歐氏距離
    k_idx = np.argsort(d)[:k]                        # 最近 k 個鄰居
    return np.bincount(ytrain[k_idx]).argmax()       # 投票

# 訓練/測試分割（每個類別前 10 張訓練、後 10 張測試）
perm = np.random.default_rng(42).permutation(20)
train_idx = np.concatenate([cls * 20 + perm[:10] for cls in range(3)])
test_idx  = np.concatenate([cls * 20 + perm[10:] for cls in range(3)])

pred = [knn(X[train_idx], y[train_idx], x) for x in X[test_idx]]
acc = np.mean(pred == y[test_idx])
print(f'kNN (k=5) 測試準確率 = {acc * 100:.1f}%  （猜謎水準是 33%）')
print('→ 特徵真的把三類紋理分開了！')

# 散佈圖：拿兩個最有感的特徵看分群（對比度 vs 游程風格代理：Laws 平滑掩膜變異數）
plt.figure(figsize=(7, 5))
sc = plt.scatter(X[:, 5], X[:, 8], c=y, cmap='Set1', s=60, alpha=0.8)
plt.colorbar(sc, ticks=[0, 1, 2], label='類別 (0=smooth,1=coarse,2=periodic)')
plt.xlabel('特徵 6：對比度 CON'); plt.ylabel('特徵 9：Laws L3×L3 變異數')
plt.title('只畫 2 個特徵就分得開 → 18 維當然更好')
plt.show()

### 專案 A 的思考題（自我驗收）
1. 把 `perm` 換掉或改 `k`（1 / 3 / 10），準確率怎麼變？為什麼 k 太小/太大都不好？
2. 如果只用「一階統計」5 個特徵（砍掉第 6–18 維），準確率掉多少？——這就是「特徵要多樣化」的實證。
3. 如果要部署到工廠的嵌入式鏡頭（運算受限），你會留哪幾個特徵？為什麼？

## 專案 B：拍手聲 vs 鋼琴聲判別器

流程：
1. 合成 10 段「鋼琴」與 10 段「拍手」（不同種子/頻率 → 同類不同樣本）
2. 每段算兩條特徵軌跡：ZCR 與頻譜通量 → 取「平均 + 變異數」
3. 用一條**手寫決策規則**（閾值）分類——先不用機器學習
4. 畫出特徵空間，證明「一条規則就夠」

In [ ]:
fs = 8000
T = 1.5

def make_sound(kind, seed):
    r = np.random.default_rng(seed)
    t = np.arange(int(fs * T)) / fs
    if kind == 'piano':                                   # 和絃：220/277/330 Hz 隨機組合 + 衰減
        f = r.choice([220, 262, 277, 330, 392], 3, replace=False)
        x = sum(np.sin(2 * np.pi * f_ * t) for f_ in f) * np.exp(-t / 2.5)
    else:                                                 # 拍手：0.3 秒一次零均值雜訊脈衝
        x = np.zeros_like(t)
        for start in np.arange(0, T, 0.3):
            i = int(start * fs); burst = int(0.025 * fs)
            env = np.exp(-np.linspace(0, 6, burst))
            x[i:i + burst] += env * r.standard_normal(burst)
    return x / np.abs(x).max()

def framing(x, N=256, hop=128):
    return np.stack([x[i:i + N] for i in range(0, len(x) - N, hop)])

def signal_features(x):
    fr = framing(x)
    N = fr.shape[1]
    zcrs = np.abs(np.diff(np.sign(fr), axis=1)).sum(axis=1) / (2 * N)   # 每幀 ZCR
    envs = (fr ** 2).mean(axis=1)                                       # 每幀能量
    return (zcrs * envs).sum() / envs.sum(), zcrs.var()                 # (能量加權 ZCR, ZCR 抖動度)

Xp = [signal_features(make_sound('piano', s)) for s in range(10)]
Xc = [signal_features(make_sound('clap',  s)) for s in range(10)]
print('piano 特徵點:', np.round(Xp, 3).T)
print('clap  特徵點:', np.round(Xc, 3).T)

In [ ]:
Xp, Xc = np.array(Xp), np.array(Xc)
plt.figure(figsize=(7.5, 5))
plt.scatter(Xp[:, 0], Xp[:, 1], c='#2563eb', s=80, label='piano 鋼琴', alpha=0.85)
plt.scatter(Xc[:, 0], Xc[:, 1], c='#ef4444', s=80, label='clap 拍手', alpha=0.85)
plt.xlabel('特徵 1：能量加權 ZCR（平均「吵雜度」）')
plt.ylabel('特徵 2：ZCR 軌跡變異數（「抖動」程度）')
plt.legend(); plt.title('兩個特徵 → 兩群完全分離！')
plt.show()

# 手寫決策規則：找一條把兩群切開的「刀」
# 觀察：clap 集中在高 ZCR + 高抖動
rule_zcr = 0.15                                          # 能量加權 ZCR 閾值
fp = (Xp[:, 0] > rule_zcr).sum()                         # 鋼琴被誤判為拍手
fc = (Xc[:, 0] <= rule_zcr).sum()                        # 拍手被誤判為鋼琴
print(f'閾值規則「ZCR > {rule_zcr} = 拍手」的錯誤數：piano 誤判 {fp} 個、clap 誤判 {fc} 個')
print(f'準確率 = {100 * (1 - (fp + fc) / 20):.0f}%  （純閾值、零機器學習！）')

### 專案 B 的思考題
1. 把拍手的脈衝間隔從 0.3s 改成 0.03s（更密集），決策規則還有效嗎？ZCR 平均會怎麼變？
2. 如果把鋼琴改成「高音和絃」（440/660/880 Hz），ZCR 會上升——閾值要往哪調？這告訴你：**特徵選擇永遠要對齊任務**。
3. 進階：把兩類的「特徵軌跡」畫成課本 Fig 7.24–7.27 的樣子，你能看出「視覺上的平穩 vs 躁動」嗎？

## 🎓 完成！你現在握有的技能清單
| # | 技能 | 實作在哪 |
|---|------|----------|
| 1 | 直方圖 5 特徵 + 看懂熵 | nb1 |
| 2 | 共生矩陣/游程/Laws 特徵（手寫 + 套件） | nb1 |
| 3 | Hu/Zernike 不變矩 + 驗證 | nb2 |
| 4 | 傅立葉描述子重建輪廓 + 鏈碼 + regionprops | nb2 |
| 5 | 切幀/窗函數/頻譜特徵/ZCR/能量/音高/MFCC | nb3 |
| 6 | **把特徵串成分類管線（即戰力）** | nb4 ← 就是這裡 |

> 下一步建議：讀課本 Ch3（分類器）→ Ch4（特徵選擇）→ 或直接跑去學 CNN/深度學習——
> 你會發現 CNN 的「卷積核」就是自動學版本的法律遮罩，「feature map」就是自動學的特徵圖。這章就是現代 AI 的鋼筋。💪